In [ ]:
import torch.nn.functional as F
import torch.nn as nn
import torch
import torch 
import torch.nn as nn 


class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_chanels, **kwargs):
        super(ConvBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_chanels, **kwargs)
        self.bn = nn.BatchNorm2d(out_chanels)

    def forward(self, x):
        return F.relu(self.bn(self.conv(x)))


class InceptionBlock(nn.Module):
    def __init__(self,  in_channels,  out_1x1, red_3x3, out_3x3, red_5x5, out_5x5, out_pool):

        super(InceptionBlock, self).__init__()
        self.branch1 = ConvBlock(in_channels, out_1x1, kernel_size=1)
        self.branch2 = nn.Sequential(
            ConvBlock(in_channels, red_3x3, kernel_size=1, padding=0),
            ConvBlock(red_3x3, out_3x3, kernel_size=3, padding=1))
        self.branch3 = nn.Sequential(
            ConvBlock(in_channels, red_5x5, kernel_size=1),
            ConvBlock(red_5x5, out_5x5, kernel_size=5, padding=2))
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, padding=1, stride=1),
            ConvBlock(in_channels, out_pool, kernel_size=1))

    def forward(self, x):
        branches = (self.branch1, self.branch2, self.branch3, self.branch4)
        return torch.cat([branch(x) for branch in branches], 1)


class InceptionAux(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(InceptionAux, self).__init__()
        #self.avgpool = nn.AdaptiveAvgPool2d(kernel_size=5, stride=3)
        self.avgpool = nn.AdaptiveAvgPool2d((4, 4))
        self.conv = ConvBlock(in_channels, 128, kernel_size=1)
        self.fc1 = nn.Linear(2048, 1024)
        self.fc2 = nn.Linear(1024, num_classes)
        self.dropout = nn.Dropout(p=0.7)

    def forward(self, x):
        x = self.avgpool(x)
        x = self.conv(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return torch.sigmoid(x) # updated added sigmoid 


class InceptionModel(nn.Module):
    def __init__(self, aux=False, residual=True, num_classes=1000):
        super(InceptionModel, self).__init__()
        self.aux = aux
        self.residual = residual

        self.conv1 = ConvBlock(3, 64, kernel_size=7, stride=2, padding=3)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.conv2 = ConvBlock(64, 192, kernel_size=3, stride=1, padding=1)

        self.incept3a = InceptionBlock(192, 64, 96, 128, 16, 32, 32)
        self.incept3b = InceptionBlock(256, 128, 128, 192, 32, 112, 80)

        self.incept4a = InceptionBlock(512, 192, 96, 208, 16, 48, 64)
        self.incept4b = InceptionBlock(512, 160, 112, 224, 24, 64, 64)

        if self.aux:
            self.aux_classifier = InceptionAux(512, num_classes)

        self.incept5a = InceptionBlock(512, 256, 160, 320, 32, 128, 128)
        self.incept5b = InceptionBlock(832, 128, 112, 256, 32, 64, 64)

        self.dropout = nn.Dropout(p=0.4)
        self.fc = nn.Linear(512 * 4 * 4, num_classes)

    def forward(self, x):

        x = self.conv1(x)
        x = self.maxpool(x)

        x = self.conv2(x)
        x = self.maxpool(x)

        x = self.incept3a(x)
        x = self.incept3b(x)

        residual = self.maxpool(x)

        x = self.incept4a(residual)

        x = self.incept4b(x)

        if self.residual:
            x += residual

        if self.aux and self.training:
            aux_out = self.aux_classifier(x)

        residual = self.maxpool(x)

        x = self.incept5a(residual)
        x = self.incept5b(x)

        if self.residual:
            x += residual

        x = F.adaptive_avg_pool2d(x, (4, 4))

        x = x.reshape(x.shape[0], -1)
        x = self.dropout(x)
        x = self.fc(x)
        x = torch.sigmoid(x)    # added sigmoid 

        if self.aux and self.training:
            return x, aux_out
        return x

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
from torchvision import transforms
from skimage import io, transform
import os
import torch.optim as optim
import itertools
from model import InceptionModel 
import logging 

log_dir = "/home/gantumur/Documents/DL/Lab456/logs"

log_path = os.path.join(log_dir, "batch_overfitting.log")


logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.FileHandler(log_path),
        logging.StreamHandler()
    ]
)

logging.info("--------------------------------")
logging.info("Starting Batch overfitting Session")
logging.info("--------------------------------")

transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
])


class FaceLandmarksDataset(Dataset):
    def __init__(self, csv_file, img_root_dir, transform=None):
        self.landmarks_frame = []
        file = open(csv_file, "r")
        while True:
            content = file.readline()
            if not content:
                break
            self.landmarks_frame.append(content.split(","))
        file.close()
        self.landmarks_frame = self.landmarks_frame[2:]

        self.img_root_dir = img_root_dir
        self.transform = transform

    def __len__(self):
        return len(self.landmarks_frame)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()
        img_name = os.path.join(
            self.img_root_dir, self.landmarks_frame[idx][0])
        image = io.imread(img_name)
        landmarks = self.landmarks_frame[idx][1:]
        landmarks = np.array([landmarks], dtype=float).reshape(-1, 2)
        scaled_landmarks = landmarks.copy()
        scaled_landmarks[:, 0], scaled_landmarks[:,
                                                 1] = scaled_landmarks[:, 0] / 178, scaled_landmarks[:, 1] / 218
        if self.transform:
            image_tr = self.transform(image)
        scaled_landmarks = scaled_landmarks.reshape(-1)
        sample = {'image': image, 'image_tr': image_tr,
                  'landmarks': landmarks, 'scaled_landmarks': scaled_landmarks}

        return sample


#1. Data Loading 
full_dataset = FaceLandmarksDataset("/home/gantumur/Documents/DL/Lab456/data/list_landmarks_align_celeba.csv",
                                    "/home/gantumur/Documents/DL/Lab456/data/img_align_celeba/img_align_celeba", transform=transform)

train_val_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_val_size
train_val_set, test_set = torch.utils.data.random_split(
    full_dataset, [train_val_size, test_size])

train_size = int(0.8 * len(train_val_set))
val_size = len(train_val_set) - train_size
train_set, val_set = torch.utils.data.random_split(
    train_val_set, [train_size, val_size])


train_loader = torch.utils.data.DataLoader(
    train_set,
    batch_size=16,
    shuffle=True # should be False fatal mistake 
)

batch_dict = next(iter(train_loader))

X_single_batch = batch_dict["image_tr"]
y_single_batch = batch_dict["scaled_landmarks"]


learning_rates = [1e-3, 5e-4, 1e-4, 1e-5]
optimizers = ["Adam", "SGD_Momentum"]

loss_functions = {
    "MSE": nn.MSELoss(),
    "SmoothL1": nn.SmoothL1Loss()
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

X_single_batch = X_single_batch.to(device).float()
y_single_batch = y_single_batch.to(device).float()

epochs_per_test = 300
best_loss = float("inf")
best_combo = None

for lr, opt_name, loss_name in itertools.product(learning_rates, optimizers, loss_functions):
    model = InceptionModel(aux=True, residual=True, num_classes=10).to(device)
    model.train()

    if opt_name == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=lr)
    else:
        optimizer = optim.AdamW(model.parameters(), lr=lr)

    criterion = loss_functions[loss_name]
    aux_weight = 0.3
    final_loss = 0.0

    for epoch in range(epochs_per_test):
        optimizer.zero_grad()
        main_out, aux_out = model(X_single_batch)

        loss_main = criterion(main_out, y_single_batch)
        loss_aux = criterion(aux_out, y_single_batch)

        total_loss = loss_main + (aux_weight * loss_aux)

        total_loss.backward()
        optimizer.step()

        final_loss = total_loss.item()
    print(
        f"Tested: LR={lr}, Opt={opt_name}, Loss={loss_name} | Final Loss: {final_loss:.6f}")
    
    logging.info(
        f"Tested: LR={lr}, Opt={opt_name}, Loss={loss_name} | Final Loss: {final_loss:.6f}")
    
    if final_loss < best_loss:
        best_loss = final_loss
        best_combo = {'LR': lr, 'Optimizer': opt_name, 'Loss': loss_name}

    del model
    del optimizer
    torch.cuda.empty_cache()

logging.info(f"Best Combination Found: {best_combo}")
logging.info(f"Lowest Loss Achieved: {best_loss:.6f}")

print(f"Best Combination Found: {best_combo}")
print(f"Lowest Loss Achieved: {best_loss:.6f}")

In [ ]:
import torch
import torch.nn as nn 
import torchvision
import numpy as np
from torch.utils.data import Dataset, DataLoader
from skimage import io, transform
import os
import torch.optim as optim
from model import InceptionModel
import logging
import time 
import pandas as pd 


log_dir = "/home/gantumur/Documents/DL/Lab456/logs"

log_path = os.path.join(log_dir, "training.log")


logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.FileHandler(log_path),
        logging.StreamHandler()
    ]
)

logging.info("--------------------------------")
logging.info("Starting Training Session")
logging.info("--------------------------------")

transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
])


class FaceLandmarksDataset(Dataset):
    def __init__(self, csv_file, img_root_dir, transform=None):
        self.landmarks_frame = []
        file = open(csv_file, "r")
        while True:
            content = file.readline()
            if not content:
                break
            self.landmarks_frame.append(content.split(","))
        file.close()
        self.landmarks_frame = self.landmarks_frame[2:]

        self.img_root_dir = img_root_dir
        self.transform = transform

    def __len__(self):
        return len(self.landmarks_frame)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()
        img_name = os.path.join(
            self.img_root_dir, self.landmarks_frame[idx][0])
        image = io.imread(img_name)
        landmarks = self.landmarks_frame[idx][1:]
        landmarks = np.array([landmarks], dtype=float).reshape(-1, 2)
        scaled_landmarks = landmarks.copy()
        scaled_landmarks[:, 0], scaled_landmarks[:,
                                                 1] = scaled_landmarks[:, 0] / 178, scaled_landmarks[:, 1] / 218
        if self.transform:
            image_tr = self.transform(image)
        scaled_landmarks = scaled_landmarks.reshape(-1)
        sample = {'image': image, 'image_tr': image_tr,
                  'landmarks': landmarks, 'scaled_landmarks': scaled_landmarks}

        return sample


# 1. Data Loading
full_dataset = FaceLandmarksDataset("/home/gantumur/Documents/DL/Lab456/data/list_landmarks_align_celeba.csv",
                                    "/home/gantumur/Documents/DL/Lab456/data/img_align_celeba/img_align_celeba", transform=transform)

train_val_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_val_size
train_val_set, test_set = torch.utils.data.random_split(
    full_dataset, [train_val_size, test_size])

train_size = int(0.8 * len(train_val_set))
val_size = len(train_val_set) - train_size
train_set, val_set = torch.utils.data.random_split(
    train_val_set, [train_size, val_size])


train_loader = DataLoader(
    train_set,
    batch_size=64,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_set,
    batch_size=64,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)


epochs = 30
LR = 5e-4
aux_weight = 0.3
min_val_loss = float("inf")
train_losses = [] 
val_losses = []  

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
logging.info(f"Device: {device}")

model = InceptionModel(
    aux=True,
    residual=True,
    num_classes=10
).to(device)

optimizer = optim.SGD(model.parameters(), lr=LR, momentum=0.9)
criterion = nn.SmoothL1Loss()

for epoch in range(epochs):
    epoch_start = time.time() 

    model.train() 
    train_loss = 0.0 

    for batch_dict in train_loader:
        x_batch = batch_dict["image_tr"].to(device).float()
        y_batch = batch_dict["scaled_landmarks"].to(device).float()

        optimizer.zero_grad()

        main_out, aux_out = model(x_batch)

        loss_main = criterion(main_out, y_batch)
        loss_aux = criterion(aux_out, y_batch)
        loss = loss_main + (aux_weight * loss_aux)

        train_loss += loss.item()

        loss.backward()
        optimizer.step()
    
    avg_train_loss = train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # validation 
    model.eval()
    val_loss = 0.0 

    with torch.no_grad():
        for batch_dict in val_loader:
            x_val = batch_dict["image_tr"].to(device).float()
            y_val = batch_dict["scaled_landmarks"].to(device).float()

            main_out = model(x_val)
            
            loss = criterion(main_out, y_val)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    epoch_time = time.time() - epoch_start 

    logging.info(f"Epoch {epoch+1}/{epochs} [{epoch_time:.1f}s]")
    logging.info(f"   Train Loss: {avg_train_loss:.4f}")
    logging.info(f"   Val   Loss: {avg_val_loss:.4f}")

    if avg_val_loss < min_val_loss:
        logging.info(f"   --> Saving Best Model ({avg_val_loss:.4f})")
        min_val_loss = avg_val_loss

        torch.save(model.state_dict(
        ), "/home/gantumur/Documents/DL/Lab456/models/best_inception_weights.pth")

logging.info("Training Complete.")

df = pd.DataFrame({
    "train_losses": train_losses,
    "val_losses": val_losses
})
df.to_csv("/home/gantumur/Documents/DL/Lab456/loss/loss_info.csv", index=False)
